In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time

driver = webdriver.Chrome()
driver.maximize_window()
wait = WebDriverWait(driver, 10)

In [ ]:
try:
    driver.get("http://localhost:5173/")
    driver.execute_script("window.localStorage.clear(); window.sessionStorage.clear();")
    driver.get("http://localhost:5173/login")
    wait.until(EC.presence_of_element_located((By.ID, "username")))

    # Submit with every field left empty (verified rules in login.jsx:
    # "Email or username is required." / "Password is required.").
    driver.find_element(By.ID, "sign-in-btn").click()
    time.sleep(2)

    u_err = driver.find_element(By.ID, "username-error").text.strip()
    p_err = driver.find_element(By.ID, "password-error").text.strip()
    print("Username message:", repr(u_err))
    print("Password message:", repr(p_err))
    assert u_err, "No required-field message for empty username."
    assert p_err, "No required-field message for empty password."

    # Empty form must not be accepted
    assert driver.find_elements(By.ID, "username"), "Login form disappeared."
    token = driver.execute_script("return window.localStorage.getItem('pharvo_access_token');")
    assert not token, "Session was created from an empty form."
    print("Empty submission correctly rejected, no session created.")
    print("PASS: Empty form submission handled")
except Exception as e:
    print("FAIL:", e)
    driver.save_screenshot("52_empty_form_FAIL.png")
finally:
    driver.quit()